# Random Training – Setup Section (Differences vs Optimized Version)

This notebook uses the same general PINN training framework as the optimized version (`04_combined_training_minimumpointnumber`)

In [ ]:
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn
from scipy.stats import qmc
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

torch.set_default_dtype(torch.float32)
dtype = torch.float32


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


device = get_device()
torch.set_default_device(device)

This section is essentially identical to `04_combined_training_minimumpointnumber`

In [ ]:
# Edge samples
N = 1000
# Number of collocation points
M = 10000
# Number of rows of the vector for temporal weighting
ROW_TEMPORAL = 5
# Number of plotted curves in the temporal weighting
NUM_CURVES = 10

# Time [s]
Z = 100.0
# Geometry [m]
L = 0.1
# Temperature [°C]
T_IC = 20.0
T_ref = 30.0
T_max = 45.0
T_min = 20.0
delta_T = T_max - T_min
# Heat flux [W/m^2]
q_x = 0.0
q_y = 10000.0
q = torch.tensor([q_x, q_y], device=device)
# Thermal conductivity [W/(m*K)]
k_l = 40.0  # In fiber direction
k_t = 20.0  # Perpendicular to fiber
# Density [kg/m^3]
rho = 2000.0
# Heat capacity at constant pressure [J/(kg*K)]
cp = 700.0
# Reference time
Z_ref = rho * cp / k_l * L**2
# Dimensionless time
Z_dimless = Z / Z_ref


# Calculate dimensionless thermal diffusivity
def rotate_thermal_conductivity(k_l, k_t, theta_deg):
    theta_rad = theta_deg * torch.pi / 180.0  # shape: [N]
    cos_theta = torch.cos(theta_rad)  # [N]
    sin_theta = torch.sin(theta_rad)  # [N]

    # Rotation matrices R: shape [N, 2, 2]
    R = torch.stack(
        [
            torch.stack([cos_theta, -sin_theta], dim=-1),
            torch.stack([sin_theta, cos_theta], dim=-1),
        ],
        dim=-2,
    )

    # K_tensor: shape [2, 2]
    K_tensor = torch.diag(torch.tensor([k_l, k_t], device=theta_deg.device))

    # Expand K_tensor to [N, 2, 2] for batch matmul
    K_tensor_batched = K_tensor.unsqueeze(0).expand(R.shape[0], -1, -1)

    # R @ K @ R^T => [N, 2, 2]
    rotated_K_tensor = R @ K_tensor_batched @ R.transpose(1, 2)

    # Inverse of each rotated K tensor
    rotated_K_tensor_inv = torch.inverse(rotated_K_tensor)

    return rotated_K_tensor, rotated_K_tensor_inv


def calculate_thermal_diffusivity_params(rotated_K_tensor, rho, cp, L, Z_ref):
    # Take first element: shape [2, 2]
    K = rotated_K_tensor[0]

    alpha_xx = K[0, 0] / (rho * cp) * Z_ref / L**2
    alpha_yy = K[1, 1] / (rho * cp) * Z_ref / L**2
    alpha_xy = K[0, 1] / (rho * cp) * Z_ref / L**2
    return alpha_xx, alpha_yy, alpha_xy


# Epochs
EPOCHS = 300
# Learning rate
LR = 0.001
# Scheduler step width
STEP = 1000
# Gamma factor of scheduler
GAMMA = 0.9
# Number of hidden neurons
HN = 60
# Number of hidden layers
LAYERS = 6
# Variance for Random Fourier Features
SIGMA = 1.0
# Number of Fourier Features
FEATURES = 40
# Initial weight of PDE Loss
W_PDE = 1.0
# Initial weight of Neumann loss
W_NEU = 1.0
# Initial weight of Initial Condition loss
W_IC = 1.0
# Weight update factor
ALPHA = 0.9
# Slope of temporal weights
epsilon = 1.0

# Sampling Strategy

This version of `sample_domain()` is structurally similar to the combined training setup. The key difference is how the **fiber orientation variable (φ)** is handled.

# Random fiber orientation per point

Instead of using a fixed or per-configuration angle, this setup samples **φ randomly at every point**:

In [ ]:
def sample_domain():
    # Top points
    x_top = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    y_top = torch.ones((N, 1), requires_grad=True, dtype=dtype)
    t_top = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    phi_top = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    top = torch.column_stack([x_top, y_top, t_top, phi_top])
    top = top[torch.argsort(top[:, 2])]

    # Bottom points
    x_bottom = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    y_bottom = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    t_bottom = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    phi_bottom = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    bottom = torch.column_stack([x_bottom, y_bottom, t_bottom, phi_bottom])
    bottom = bottom[torch.argsort(bottom[:, 2])]

    # Left points
    x_left = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    y_left = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    t_left = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    phi_left = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    # Exponential scaling
    y_left_exp = (torch.exp(-3 * y_left) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    left = torch.column_stack([x_left, y_left_exp, t_left, phi_left])
    left = left[torch.argsort(left[:, 2])]

    # Right points
    x_right = torch.ones((N, 1), requires_grad=True, dtype=dtype)
    y_right = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    t_right = Z_dimless * torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    phi_right = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    # Exponential scaling
    y_right_exp = (torch.exp(-3 * y_right) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    right = torch.column_stack([x_right, y_right_exp, t_right, phi_right])
    right = right[torch.argsort(right[:, 2])]

    # Collocation points
    points = qmc.LatinHypercube(d=3).random(M)
    # Sort the points based on the third dimension (time)
    points_sorted = points[points[:, 2].argsort()]
    x_collocation = torch.tensor(points_sorted[:, 0], requires_grad=True, dtype=dtype)
    y_collocation = torch.tensor(points_sorted[:, 1], requires_grad=True, dtype=dtype)
    t_collocation = Z_dimless * torch.tensor(
        points_sorted[:, 2], requires_grad=True, dtype=dtype
    )
    phi_collocation = torch.tensor(
        qmc.LatinHypercube(d=1).random(M), requires_grad=True, dtype=dtype
    )
    # Exponential scaling
    y_collo_exp = (torch.exp(-3 * y_collocation) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    collocation = torch.column_stack(
        [x_collocation, y_collo_exp, t_collocation, phi_collocation]
    )

    # Initial points
    rand_samp = qmc.LatinHypercube(d=2).random(N)
    x_t0 = torch.tensor(rand_samp[:, 0], requires_grad=True, dtype=dtype)
    y_t0 = torch.tensor(rand_samp[:, 1], requires_grad=True, dtype=dtype)
    y_t0_exp = (torch.exp(-3 * y_t0) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    t_t0 = torch.zeros((N, 1), requires_grad=True, dtype=dtype)
    phi_t0 = torch.tensor(
        qmc.LatinHypercube(d=1).random(N), requires_grad=True, dtype=dtype
    )
    # Exponential scaling
    y_0_exp = (torch.exp(-3 * y_t0) - 1) / (
        torch.exp(torch.tensor(-3.0, dtype=dtype)) - 1
    )
    initial = torch.column_stack([x_t0, y_0_exp, t_t0, phi_t0])

    return top, bottom, left, right, collocation, initial


top, bottom, left, right, collocation, initial = sample_domain()

# 3D Sampling Visualization

This visualization is structurally identical to the combined training version.

In [ ]:
def plot_interactive_3D():
    top_np = top.detach().cpu().numpy()
    bottom_np = bottom.detach().cpu().numpy()
    left_np = left.detach().cpu().numpy()
    right_np = right.detach().cpu().numpy()
    collocation_np = collocation.detach().cpu().numpy()
    initial_np = initial.detach().cpu().numpy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=top_np[:, 0],
            y=top_np[:, 1],
            z=top_np[:, 2],
            mode="markers",
            marker=dict(
                size=4,
                color=top_np[:, 3],
                colorscale="Viridis",
                colorbar=dict(title="θ"),
            ),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=bottom_np[:, 0],
            y=bottom_np[:, 1],
            z=bottom_np[:, 2],
            mode="markers",
            marker=dict(size=4, color=bottom_np[:, 3], colorscale="Viridis"),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=left_np[:, 0],
            y=left_np[:, 1],
            z=left_np[:, 2],
            mode="markers",
            marker=dict(size=4, color=left_np[:, 3], colorscale="Viridis"),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=right_np[:, 0],
            y=right_np[:, 1],
            z=right_np[:, 2],
            mode="markers",
            marker=dict(size=4, color=right_np[:, 3], colorscale="Viridis"),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=collocation_np[:, 0],
            y=collocation_np[:, 1],
            z=collocation_np[:, 2],
            mode="markers",
            marker=dict(size=3, color=collocation_np[:, 3], colorscale="Viridis"),
        )
    )
    fig.add_trace(
        go.Scatter3d(
            x=initial_np[:, 0],
            y=initial_np[:, 1],
            z=initial_np[:, 2],
            mode="markers",
            marker=dict(size=4, color=initial_np[:, 3], colorscale="Viridis"),
        )
    )
    fig.update_layout(
        title="Interactive 3D of Sampled Points",
        scene=dict(
            xaxis_title="x [-]",
            yaxis_title="y [-]",
            zaxis_title="t[-]",
            aspectmode="cube",
        ),
        margin=dict(l=0, r=0, b=0, t=40),
    )

    fig.show()


plot_interactive_3D()

# Neural Network

The network architecture is identical to `04_combined_training_minimumpointnumber`. Only the interpretation of the input space differs.


# Input difference (key change)

The network now treats fiber orientation **φ as a continuous input variable**:

In [ ]:
# Define the neural network


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.B = torch.normal(0.0, SIGMA, size=(4, FEATURES), device=device)
        self.layers = nn.ModuleList([nn.Linear(2 * FEATURES, HN)])

        for _ in range(LAYERS - 1):
            self.layers.append(nn.Linear(HN, HN))

        # Output layer
        self.output_layer = nn.Linear(HN, 1)

    def forward(self, x, y, t, phi):
        inputs = torch.column_stack([x, y, t, phi])

        pi = torch.tensor(np.pi, dtype=dtype, device=device)
        features = torch.cat(
            [torch.sin(2 * pi * inputs @ self.B), torch.cos(2 * pi * inputs @ self.B)],
            dim=-1,
        )

        for layer in self.layers:
            features = torch.tanh(layer(features))

        output = self.output_layer(features)

        return output


net = Net()

# PDE Residual and Temporal Loss

This block is identical in structure to `04_combined_training_minimumpointnumber`. The only meaningful change is that the PDE now depends on a **continuous random fiber orientation φ** instead of a fixed or scenario-based set.

# PDE residual

The anisotropic heat equation is evaluated via automatic differentiation:

In [ ]:
def pde_residual(x, y, t, phi):
    T = net(x, y, t, phi)

    T_x = torch.autograd.grad(T.sum(), x, create_graph=True, retain_graph=True)[0]
    T_xx = torch.autograd.grad(T_x.sum(), x, create_graph=True)[0]
    T_y = torch.autograd.grad(T.sum(), y, create_graph=True, retain_graph=True)[0]
    T_yy = torch.autograd.grad(T_y.sum(), y, create_graph=True)[0]
    T_xy = torch.autograd.grad(T_x.sum(), y, create_graph=True)[0]
    T_t = torch.autograd.grad(T.sum(), t, create_graph=True, retain_graph=True)[0]
    # Calculate rotated tensor and alpha components
    phi_deg = phi * 180.0
    rotated_K_tensor, rotated_K_tensor_inv = rotate_thermal_conductivity(
        k_l, k_t, phi_deg
    )
    alpha_xx, alpha_yy, alpha_xy = calculate_thermal_diffusivity_params(
        rotated_K_tensor, rho, cp, L, Z_ref
    )
    # Calculate the PDE residual
    residual_T = T_t - (alpha_xx * T_xx + alpha_yy * T_yy + 2 * alpha_xy * T_xy)
    return residual_T


def compute_pde_losses_mean(pde_losses):
    total_len = len(pde_losses)
    pde_matrix = pde_losses.reshape(
        ROW_TEMPORAL, total_len // ROW_TEMPORAL
    )  # Reshape the pde_losses to a matrix
    pde_losses_mean = torch.mean(pde_matrix, dim=1)  # Mean of the rows
    return pde_losses_mean


def compute_weighted_pde_loss(pde_losses_mean):
    # Calculate the temporal weights
    cumsum = torch.cumsum(
        pde_losses_mean, dim=0
    )  # Vector cumsum where the components are the accumulative sum of the losses
    cumsum_shifted = torch.roll(
        cumsum, shifts=1, dims=0
    )  # Shift right to get sum up to i-1
    cumsum_shifted[0] = 0.0  # Fix first value to represent empty sum
    wtemp = torch.exp(-epsilon * cumsum_shifted)

    # Calculate the weighted pde loss
    weighted_pde_losses = wtemp * pde_losses_mean
    weighted_pde_loss = torch.sum(weighted_pde_losses) / ROW_TEMPORAL
    return wtemp, weighted_pde_losses, weighted_pde_loss


wtemp_evolution = []
pde_losses_mean_evolution = []
weighted_pde_losses_evolution = []

This section is essentially identical to `04_combined_training_minimumpointnumber`

In [ ]:
# Define the MSE Loss function
mse = torch.nn.MSELoss()


def make_grad_tensor(tensor):
    return tensor.requires_grad_(True)


def compute_loss(top, bottom, left, right, collocation, initial):
    # Ensure tensors are properly set up for differentiation
    x_top = top[:, 0]
    y_top = top[:, 1]
    t_top = top[:, 2]
    phi_top = top[:, 3]
    x_bottom = bottom[:, 0]
    y_bottom = bottom[:, 1]
    t_bottom = bottom[:, 2]
    phi_bottom = bottom[:, 3]
    x_left = left[:, 0]
    y_left = left[:, 1]
    t_left = left[:, 2]
    phi_left = left[:, 3]
    x_right = right[:, 0]
    y_right = right[:, 1]
    t_right = right[:, 2]
    phi_right = right[:, 3]

    # Compute predicted values
    pred_top = net(x_top, y_top, t_top, phi_top)
    pred_bottom = net(x_bottom, y_bottom, t_bottom, phi_bottom)
    pred_left = net(x_left, y_left, t_left, phi_left)
    pred_right = net(x_right, y_right, t_right, phi_right)

    # Compute the derivatives
    dpred_dx_top = torch.autograd.grad(
        pred_top.sum(), x_top, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_top = torch.autograd.grad(
        pred_top.sum(), y_top, create_graph=True, retain_graph=True
    )[0]
    dpred_dy_bottom = torch.autograd.grad(
        pred_bottom.sum(), y_bottom, create_graph=True, retain_graph=True
    )[0]
    dpred_dx_left = torch.autograd.grad(
        pred_left.sum(), x_left, create_graph=True, retain_graph=True
    )[0]
    dpred_dx_right = torch.autograd.grad(
        pred_right.sum(), x_right, create_graph=True, retain_graph=True
    )[0]

    # Neumann boundary conditions
    real_phi_top = phi_top * 180.0  # Convert to degrees
    rotated_k_tensor, rotated_K_tensor_inv = rotate_thermal_conductivity(
        k_l, k_t, real_phi_top
    )
    q_expanded = q.unsqueeze(0).expand(
        rotated_K_tensor_inv.shape[0], -1
    )  # Expand q to match the batch size
    # Calculate the change in temperature across the top boundary
    dU_top = (rotated_K_tensor_inv @ q_expanded.unsqueeze(-1)).squeeze(-1) * (
        L / delta_T
    )  # Shape: [100, 2]
    dU_dx_top = dU_top[:, 0] * torch.ones_like(dpred_dx_top)
    dU_dy_top = dU_top[:, 1] * torch.ones_like(dpred_dy_top)
    dU_dy_bottom = torch.zeros_like(dpred_dy_bottom)
    dU_dx_left = torch.zeros_like(dpred_dx_left)
    dU_dx_right = torch.zeros_like(dpred_dx_right)

    # Compute the individual boundary losses using MSE
    loss_top_x = mse(dpred_dx_top, dU_dx_top)
    loss_top_y = mse(dpred_dy_top, dU_dy_top)
    loss_bottom = mse(dpred_dy_bottom, dU_dy_bottom)
    loss_left = mse(dpred_dx_left, dU_dx_left)
    loss_right = mse(dpred_dx_right, dU_dx_right)
    neumann_losses = loss_top_x + loss_top_y + loss_bottom + loss_left + loss_right

    # Initial condition
    pred_ic = net(initial[:, 0], initial[:, 1], initial[:, 2], initial[:, 3])
    T_ic = T_IC * torch.ones_like(initial[:, 0]).unsqueeze(1)
    U_ic = (T_ic - T_ref) / delta_T
    initial_losses = mse(pred_ic, U_ic)

    # PDE residual loss
    # Loss for initial and collocation points
    initial_collo = torch.cat([initial, collocation], dim=0)
    pred_pde = pde_residual(
        initial_collo[:, 0],
        initial_collo[:, 1],
        initial_collo[:, 2],
        initial_collo[:, 3],
    )
    pde_losses = pred_pde**2

    return initial_losses, neumann_losses, pde_losses


w_pde_history = []
w_neu_history = []
w_ic_history = []


def compute_gradient_norm(loss):
    grads = torch.autograd.grad(loss, tuple(net.parameters()), allow_unused=True)
    return sum(0 if grad is None else torch.linalg.norm(grad) for grad in grads)


def update_weight(weight, grad_sum, norm):
    new_weight = grad_sum / norm
    return ALPHA * weight + (1 - ALPHA) * new_weight

# Training loop
Randomized sampling at every epoch

In [ ]:
# Train PINN
history = []
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
scheduler = StepLR(optimizer, step_size=STEP, gamma=GAMMA)

print("Training...")
for epoch in tqdm(range(EPOCHS)):
    top, bottom, left, right, collocation, initial = sample_domain()

    # Train batches of 100 collocation points
    for collo in torch.chunk(collocation, int(M / 100)):
        optimizer.zero_grad()

        initial_l, neumann_l, pde_l = compute_loss(
            top, bottom, left, right, collocation, initial
        )

        pde_loss_mean = compute_pde_losses_mean(pde_l)
        wtemp, weighted_pde_losses, pde_loss_weighted = compute_weighted_pde_loss(
            pde_loss_mean
        )
        wtemp_evolution.append(wtemp.detach().cpu().numpy())
        pde_losses_mean_evolution.append(pde_loss_mean.detach().cpu().numpy())
        weighted_pde_losses_evolution.append(weighted_pde_losses.detach().cpu().numpy())

        loss = W_IC * initial_l + W_NEU * neumann_l + W_PDE * pde_loss_weighted
        loss.backward(retain_graph=True)
        optimizer.step()

        w_ic_history.append(W_IC)
        w_neu_history.append(W_NEU)
        w_pde_history.append(W_PDE)

        scheduler.step()
        history.append(loss.item())

    # Rebalance weights every 5 epochs
    if epoch % 5 == 0:
        initial_l, neumann_l, pde_l = compute_loss(
            top, bottom, left, right, collocation, initial
        )
        pde_loss_mean = compute_pde_losses_mean(pde_l)
        wtemp, weighted_pde_losses, pde_loss_weighted = compute_weighted_pde_loss(
            pde_loss_mean
        )
        grad_ic = compute_gradient_norm(initial_l)
        grad_neu = compute_gradient_norm(neumann_l)
        grad_pde = compute_gradient_norm(pde_loss_weighted)
        grad_sum = grad_ic + grad_neu + grad_pde
        W_IC = update_weight(W_IC, grad_sum, grad_ic)
        W_NEU = update_weight(W_NEU, grad_sum, grad_neu)
        W_PDE = update_weight(W_PDE, grad_sum, grad_pde)

        print(f"Epoch {epoch}: W_IC={W_IC:.5f}, W_NEU={W_NEU:.5f}, W_PDE={W_PDE:.5f}")


# Plot training loss
plt.semilogy(history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid()
plt.title("Training Loss with Dynamic Weights")
plt.show()

The rest of the code is essentially identical to `04_combined_training_minimumpointnumber`

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 5))

# Plot evolution of W_IC W_NEU and W_PDE in the same graph
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_ic_history],
    label="W_IC",
    color="g",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_neu_history],
    label="W_NEU",
    color="b",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.plot(
    [w.cpu().item() if torch.is_tensor(w) else w for w in w_pde_history],
    label="W_PDE",
    color="r",
    linestyle="--",
    marker="s",
    markersize=3,
)
plt.xlabel("Epoch")
plt.ylabel("Weight Value")
plt.legend()
plt.grid()
plt.title("Evolution of W_IC, W_NEU and W_PDE over Epochs")

plt.show()

In [ ]:
def plot_wtemp_evolution(wtemp_evolution):
    # Convert list of vectors to numpy array
    wtemp_array = np.array(wtemp_evolution)
    total_epochs = len(wtemp_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Evolution of the components of wtemp ({num_curves} evenly spaced epochs)",
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = wtemp_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"wtemp{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of wtemp component")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_wtemp_evolution(wtemp_evolution)

In [ ]:
def plot_pdelosses_evolution(pde_losses_mean_evolution):
    # Convert list of vectors to numpy array
    pdelossesmean_array = np.array(pde_losses_mean_evolution)
    total_epochs = len(pdelossesmean_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Evolution of the components of the mean pde losses ({num_curves} evenly spaced epochs)",
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = pdelossesmean_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"pde_loss_mean{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of the mean pde losses components")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_pdelosses_evolution(pde_losses_mean_evolution)

In [ ]:
def plot_weighted_pdelosses_evolution(weighted_pde_losses_evolution):
    # Convert list of vectors to numpy array
    weightedlossesmean_array = np.array(weighted_pde_losses_evolution)
    total_epochs = len(weightedlossesmean_array)

    # Define how many curves to plot
    num_curves = NUM_CURVES

    # Evenly spaced epoch indices (rounded to int)
    epochs_to_plot = np.linspace(0, total_epochs - 1, num=num_curves, dtype=int)
    # Skip the first epoch
    epochs_to_plot = epochs_to_plot[1:]

    # Define colormap with as many colors as curves
    colormap = cm.get_cmap("viridis", num_curves)
    colors = [colormap(i) for i in range(num_curves)]

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(
        f"Evolution of the components of the weighted mean pde losses ({num_curves} evenly spaced epochs)",
        fontsize=14,
        fontweight="bold",
    )

    # Plot the selected vectors
    for idx, epoch in enumerate(epochs_to_plot):
        values = weightedlossesmean_array[epoch][:ROW_TEMPORAL]
        ax.plot(
            range(ROW_TEMPORAL),
            values,
            color=colors[idx],
            linewidth=2,
            label=f"Epoch {epoch+1}",
        )

    # X-axis settings
    ax.set_xticks(range(ROW_TEMPORAL))
    ax.set_xticklabels([f"weighted_pde_loss_mean{i+1}" for i in range(ROW_TEMPORAL)])
    ax.set_xlabel("Vector components")
    ax.set_ylabel("Value of the weighted mean pde losses components")
    ax.grid(True)

    # Add legend
    ax.legend(title="Epochs", loc="best")
    fig.tight_layout()

    # Show the figure
    plt.show()


# Call the function
plot_weighted_pdelosses_evolution(weighted_pde_losses_evolution)

In [ ]:
# Plot training loss
plt.figure(figsize=(19 / 2.54, 7 / 2.54))  # 19cm x 7cm in inches
plt.semilogy(history, color="black", linewidth=1.5)  # Black line
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.title("Training with sequential sample domains", fontsize=11)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Angles between 0 and 180 degrees


def wrap_angle(angle):
    return angle % 180


def plot_all_temperature_heatmaps(angles):
    num_points = 100
    val_x, val_y = np.meshgrid(
        np.linspace(0, 1, num_points), np.linspace(0, 1, num_points)
    )
    X_flat, Y_flat = val_x.flatten(), val_y.flatten()
    X_tensor = torch.tensor(X_flat, dtype=dtype, device=device)
    Y_tensor = torch.tensor(Y_flat, dtype=dtype, device=device)

    val_X = L * val_x
    val_Y = L * val_y

    fig, axes = plt.subplots(len(angles), 2, figsize=(12, 5 * len(angles)))
    fig.suptitle("Temperature Distribution PINN", fontsize=18, fontweight="bold")

    for i, phi in enumerate(angles):
        phi_og = phi
        phi = wrap_angle(phi_og)  # Ensure phi is in the range [0, 180]
        phi_normalized = phi / 180.0
        phi_tensor = torch.full_like(X_tensor, phi_normalized)

        t_0 = torch.zeros_like(X_tensor)
        t_Z = torch.full_like(X_tensor, Z / Z_ref)

        U_t0 = (
            net(X_tensor, Y_tensor, t_0, phi_tensor)
            .detach()
            .cpu()
            .numpy()
            .reshape(num_points, num_points)
        )
        U_tZ = (
            net(X_tensor, Y_tensor, t_Z, phi_tensor)
            .detach()
            .cpu()
            .numpy()
            .reshape(num_points, num_points)
        )

        T_t0 = U_t0 * delta_T + T_ref
        T_tZ = U_tZ * delta_T + T_ref

        ax0 = axes[i, 0] if len(angles) > 1 else axes[0]
        ax1 = axes[i, 1] if len(angles) > 1 else axes[1]

        im0 = ax0.contourf(val_X, val_Y, T_t0, cmap="viridis", levels=30)
        fig.colorbar(im0, ax=ax0, label="Temperature [°C]")
        ax0.set_title(f"φ = {phi_og}°, t=0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")
        ax0.set_aspect("equal")

        im1 = ax1.contourf(val_X, val_Y, T_tZ, cmap="viridis", levels=30)
        fig.colorbar(im1, ax=ax1, label="Temperature [°C]")
        ax1.set_title(f"φ = {phi_og}°, t={Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")
        ax1.set_aspect("equal")

    plt.tight_layout(rect=(0, 0.03, 1, 0.95))


plot_all_temperature_heatmaps(
    [-60.0, -45.0, -30.0, 0.0, 30.0, 45.0, 60.0, 90.0, 120.0, 135.0, 150.0, 180.0]
)

In [ ]:
# Path to project root: anisotropic-heat-transfer-pinns
project_root = Path.cwd().parent

# Path to comsol_data folder
comsol_data_path = project_root / "comsol_data"


comsoldataminus60 = pd.read_csv(
    comsol_data_path / "ccsicplate-60deg01m20c10000w100s.csv"
)
comsoldataminus45 = pd.read_csv(
    comsol_data_path / "ccsicplate-45deg01m20c10000w100s.csv"
)
comsoldataminus30 = pd.read_csv(
    comsol_data_path / "ccsicplate-30deg01m20c10000w100s.csv"
)
comsoldata0 = pd.read_csv(comsol_data_path / "ccsicplate0deg01m20c10000w100s.csv")
comsoldata30 = pd.read_csv(comsol_data_path / "ccsicplate30deg01m20c10000w100s.csv")
comsoldata45 = pd.read_csv(comsol_data_path / "ccsicplate45deg01m20c10000w100s.csv")
comsoldata60 = pd.read_csv(comsol_data_path / "ccsicplate60deg01m20c10000w100s.csv")
comsoldata90 = pd.read_csv(comsol_data_path / "ccsicplate90deg01m20c10000w100s.csv")

In [ ]:
def plot_all_error_heatmaps(angle_data_map):
    fig, axes = plt.subplots(
        len(angle_data_map), 2, figsize=(12, 5 * len(angle_data_map))
    )
    fig.suptitle(
        "Scaled Relative Temperature Error [%]", fontsize=18, fontweight="bold"
    )

    for i, (theta, comsoldata) in enumerate(angle_data_map.items()):
        X_comsol = torch.tensor(comsoldata.iloc[:, 0].values, dtype=dtype)
        Y_comsol = torch.tensor(comsoldata.iloc[:, 1].values, dtype=dtype)
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        theta_og = theta
        theta = wrap_angle(theta)
        theta_norm = theta / 180
        theta_tensor = torch.full_like(x_comsol, theta_norm, dtype=dtype)

        t_0 = torch.zeros_like(x_comsol)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_t0 = (
            net(x_comsol, y_comsol, t_0, theta_tensor).detach().cpu() * delta_T + T_ref
        )
        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, theta_tensor).detach().cpu() * delta_T + T_ref
        )

        T_comsol_t0 = torch.tensor(comsoldata.iloc[:, 2].values).unsqueeze(1).cpu()
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_t0 = ((T_pinn_t0 - T_comsol_t0) / T_comsol_t0 * 100).squeeze()
        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()

        ax0 = axes[i, 0] if len(angle_data_map) > 1 else axes[0]
        ax1 = axes[i, 1] if len(angle_data_map) > 1 else axes[1]

        sc0 = ax0.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_t0.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
        )
        fig.colorbar(sc0, ax=ax0, label="Error [%]")
        ax0.set_title(f"θ = {theta_og}°, t=0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")

        sc1 = ax1.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_tZ.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
        )
        fig.colorbar(sc1, ax=ax1, label="Error [%]")
        ax1.set_title(f"θ = {theta_og}°, t={Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")

    plt.tight_layout(rect=(0, 0.03, 1, 0.95))


angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}
plot_all_error_heatmaps(angle_data_map)

In [ ]:
def plot_all_error_heatmaps(angle_data_map):
    # First pass: determine global color scale
    all_error_tZ = []
    angle_errors = {}  # Store errors per angle

    for phi, comsoldata in angle_data_map.items():
        X_comsol = torch.tensor(
            comsoldata.iloc[:, 0].values, dtype=dtype, device=device
        )
        Y_comsol = torch.tensor(
            comsoldata.iloc[:, 1].values, dtype=dtype, device=device
        )
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_og = phi
        phi = wrap_angle(phi_og)
        phi_norm = phi / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu() * delta_T + T_ref
        )
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()
        angle_errors[phi_og] = error_tZ.cpu()
        all_error_tZ.append(error_tZ.cpu())

    # Combine all errors and find symmetric color limit
    all_error_tZ = torch.cat(all_error_tZ)
    vmax = all_error_tZ.abs().max().item()
    vmin = -vmax
    print(f"\nGlobal color scale limits: vmin={vmin:.2f}%, vmax={vmax:.2f}%")

    # Find angles with max and min of the maximal absolute errors
    max_angle = max(
        angle_errors.keys(), key=lambda k: angle_errors[k].abs().max().item()
    )
    max_error_value = angle_errors[max_angle].abs().max().item()

    min_angle = min(
        angle_errors.keys(), key=lambda k: angle_errors[k].abs().max().item()
    )
    min_error_value = angle_errors[min_angle].abs().max().item()

    print(f"Maximum absolute error = {max_error_value:.3f}% found at φ = {max_angle}°")
    print(
        f"Smallest maximum absolute error = {min_error_value:.3f}% found at φ = {min_angle}°\n"
    )

    # Second pass: generate plots
    fig, axes = plt.subplots(
        len(angle_data_map), 2, figsize=(12, 5 * len(angle_data_map))
    )
    fig.suptitle(
        "Scaled Relative Temperature Error [%]", fontsize=18, fontweight="bold"
    )

    for i, (phi, comsoldata) in enumerate(angle_data_map.items()):
        X_comsol = torch.tensor(
            comsoldata.iloc[:, 0].values, dtype=dtype, device=device
        )
        Y_comsol = torch.tensor(
            comsoldata.iloc[:, 1].values, dtype=dtype, device=device
        )
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_og = phi
        phi = wrap_angle(phi_og)
        phi_norm = phi / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)

        t_0 = torch.zeros_like(x_comsol)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_t0 = (
            net(x_comsol, y_comsol, t_0, phi_tensor).detach().cpu() * delta_T + T_ref
        )
        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu() * delta_T + T_ref
        )

        T_comsol_t0 = torch.tensor(comsoldata.iloc[:, 2].values).unsqueeze(1).cpu()
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_t0 = ((T_pinn_t0 - T_comsol_t0) / T_comsol_t0 * 100).squeeze()
        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()

        # Compute both mean and max absolute errors for this angle
        mean_error_tZ = error_tZ.abs().mean().item()
        max_error_tZ = error_tZ.abs().max().item()

        print(
            f"φ = {phi_og:6.1f}° → Mean |Error| = {mean_error_tZ:.3f}%,  Max |Error| = {max_error_tZ:.3f}%"
        )

        ax0 = axes[i, 0] if len(angle_data_map) > 1 else axes[0]
        ax1 = axes[i, 1] if len(angle_data_map) > 1 else axes[1]

        sc0 = ax0.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_t0.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(sc0, ax=ax0, label="Error [%]")
        ax0.set_title(f"φ = {phi_og}°, t=0 [s]")
        ax0.set_xlabel("x [m]")
        ax0.set_ylabel("y [m]")

        sc1 = ax1.scatter(
            X_comsol.cpu().numpy(),
            Y_comsol.cpu().numpy(),
            c=error_tZ.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
            vmin=vmin,
            vmax=vmax,
        )
        fig.colorbar(sc1, ax=ax1, label="Error [%]")
        ax1.set_title(f"φ = {phi_og}°, t={Z} [s]")
        ax1.set_xlabel("x [m]")
        ax1.set_ylabel("y [m]")

    plt.tight_layout(rect=(0, 0.03, 1, 0.95))
    plt.show()


angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}
plot_all_error_heatmaps(angle_data_map)

In [ ]:
def plot_extreme_error_heatmaps(angle_data_map):
    # First pass: compute errors and MRE per angle
    angle_errors = {}
    mean_errors = {}

    for phi, comsoldata in angle_data_map.items():
        X_comsol = torch.tensor(
            comsoldata.iloc[:, 0].values, dtype=dtype, device=device
        )
        Y_comsol = torch.tensor(
            comsoldata.iloc[:, 1].values, dtype=dtype, device=device
        )
        x_comsol = X_comsol / L
        y_comsol = Y_comsol / L
        phi_bereich = wrap_angle(phi)
        phi_norm = phi_bereich / 180.0
        phi_tensor = torch.full_like(x_comsol, phi_norm)
        t_Z = torch.full_like(x_comsol, Z / Z_ref)

        T_pinn_tZ = (
            net(x_comsol, y_comsol, t_Z, phi_tensor).detach().cpu() * delta_T + T_ref
        )
        T_comsol_tZ = torch.tensor(comsoldata.iloc[:, 12].values).unsqueeze(1).cpu()

        error_tZ = ((T_pinn_tZ - T_comsol_tZ) / T_comsol_tZ * 100).squeeze()
        angle_errors[phi] = error_tZ
        mean_errors[phi] = error_tZ.abs().mean().item()

    # Identify angles with highest and lowest MRE
    max_mre_angle = max(mean_errors, key=lambda angle: mean_errors[angle])
    min_mre_angle = min(mean_errors, key=lambda angle: mean_errors[angle])

    print(
        f"Angle with highest MRE: {max_mre_angle}° → {mean_errors[max_mre_angle]:.3f}%"
    )
    print(
        f"Angle with lowest MRE: {min_mre_angle}° → {mean_errors[min_mre_angle]:.3f}%"
    )

    # Determine global color scale based on combined errors
    all_errors = torch.cat([angle_errors[max_mre_angle], angle_errors[min_mre_angle]])
    vmax = all_errors.abs().max().item()
    vmin = -vmax
    print(f"Global color scale: vmin={vmin:.2f}%, vmax={vmax:.2f}%")

    # Generate separate plots for min and max MRE
    for phi in [min_mre_angle, max_mre_angle]:
        phi_bereich = wrap_angle(phi)
        comsoldata = angle_data_map[phi]
        X_comsol = torch.tensor(comsoldata.iloc[:, 0].values)
        Y_comsol = torch.tensor(comsoldata.iloc[:, 1].values)
        error_tZ = angle_errors[phi]

        x_dimless = X_comsol / L
        y_dimless = Y_comsol / L

        # Convert cm to inches for figsize
        fig_width = 8 / 2.54
        fig_height = 6 / 2.54

        fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)
        sc = ax.scatter(
            x_dimless.cpu().numpy(),
            y_dimless.cpu().numpy(),
            c=error_tZ.cpu().numpy(),
            cmap="coolwarm",
            marker="o",
            vmin=vmin,
            vmax=vmax,
        )
        cbar = fig.colorbar(sc, ax=ax, label="RE [%]")
        cbar.ax.tick_params(labelsize=11)
        cbar.ax.yaxis.label.set_fontsize(11)

        ax.set_title(f"φ = {phi_bereich}°", fontsize=11)
        ax.set_xlabel("x [-]", fontsize=11)
        ax.set_ylabel("y [-]", fontsize=11)

        # Axis limits based on physical coordinates
        ax.set_xlim(x_dimless.min().item(), x_dimless.max().item())
        ax.set_ylim(y_dimless.min().item(), y_dimless.max().item())

        # 3 ticks per axis
        ax.set_xticks(
            [
                x_dimless.min().item(),
                (x_dimless.min().item() + x_dimless.max().item()) / 2,
                x_dimless.max().item(),
            ]
        )
        ax.set_yticks(
            [
                y_dimless.min().item(),
                (y_dimless.min().item() + y_dimless.max().item()) / 2,
                y_dimless.max().item(),
            ]
        )
        ax.tick_params(axis="both", labelsize=11)

        plt.show()


angle_data_map = {
    -60.0: comsoldataminus60,
    -45.0: comsoldataminus45,
    -30.0: comsoldataminus30,
    0.0: comsoldata0,
    30.0: comsoldata30,
    45.0: comsoldata45,
    60.0: comsoldata60,
    90.0: comsoldata90,
    120.0: comsoldataminus60,
    135.0: comsoldataminus45,
    150.0: comsoldataminus30,
    180.0: comsoldata0,
}

plot_extreme_error_heatmaps(angle_data_map)